# 04 — Orchestration (single route: Bronze → Silver → Gold)

Runs the three stages back-to-back in one notebook (serverless compute does
not support `dbutils.notebook.run`, so the stages execute in-process) and logs
every stage to `workspace.gold.pipeline_run_log` — the same table the
Lakeflow job uses. Any stage failure is logged and aborts the run.

In [0]:
# --- Bootstrap src/ import + run-id resolution + stage logging ---
import datetime
import sys
import uuid

from databricks.sdk.runtime import dbutils

try:
    _nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().value()
    _root = "/" + "/".join(_nb.strip("/").split("/")[:-2])
except Exception:
    _root = "/Users/sreekanthreddy2000@gmail.com/clinical-trial-dq-pip"
sys.path.insert(0, "/Workspace" + _root)

RUN_LOG_TABLE = "workspace.gold.pipeline_run_log"

def _pipeline_run_id():
    # 1) Job parameter (run_id={{run_id}} on the job) via widget
    try:
        dbutils.widgets.text("run_id", "")
        rid = dbutils.widgets.get("run_id")
        if rid:
            return rid
    except Exception:
        pass
    # 2) Notebook task context inside a job run
    try:
        import json
        ctx = json.loads(dbutils.notebook.entry_point.getDbutils().notebook().getContext().toJson())
        rid = ctx.get("jobRunId") or ctx.get("id")
        if rid:
            return str(rid)
    except Exception:
        pass
    # 3) Spark-conf based detection (non-serverless job clusters)
    for key in ("spark.databricks.job.runId", "spark.jobRunId", "spark.jobGroup.id"):
        try:
            rid = spark.conf.get(key)
            if rid:
                return rid
        except Exception:
            pass
    return "adhoc-" + uuid.uuid4().hex[:8]  # fallback for interactive runs

RUN_ID = _pipeline_run_id()

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {RUN_LOG_TABLE} (
        run_id STRING, stage_name STRING, start_time TIMESTAMP, end_time TIMESTAMP,
        status STRING, records_processed BIGINT, error_message STRING
    ) USING DELTA
""")

def _log_stage(status, records=None, error=None):
    end_ts = datetime.datetime.now(datetime.UTC)
    err = None if error is None else f"{STAGE_NAME} failed: {type(error).__name__}: {error}"
    spark.createDataFrame(
        [(RUN_ID, STAGE_NAME, STAGE_START_TS, end_ts, status, records, err)],
        "run_id string, stage_name string, start_time timestamp, end_time timestamp, "
        "status string, records_processed bigint, error_message string",
    ).createOrReplaceTempView("temp_run_log_row")
    spark.sql(f"""MERGE INTO {RUN_LOG_TABLE} t USING temp_run_log_row s
                 ON t.run_id = s.run_id AND t.stage_name = s.stage_name
                 WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *""")
    msg = f"[RUN LOG] run_id={RUN_ID} | stage={STAGE_NAME} | status={status}"
    if records is not None:
        msg += f" | records={records}"
    if err:
        msg += f" | {err}"
    print(msg)

In [0]:
# --- Stage runners: thin wrappers over the src functions ---
from src.transforms import (
    BUSINESS_COLUMNS, build_site_kpi_summary, build_study_kpi_summary,
    new_rows_only, read_csv_all_strings, transform_sites, transform_studies,
    transform_subjects, transform_visits,
)
from src.dq_checks import all_passed, run_dq_checks, summarize

BRONZE, SILVER, GOLD = "workspace.bronze", "workspace.silver", "workspace.gold"
LANDING_PATH = "/Volumes/workspace/landing/raw_files"
DATASETS = [("studies", "studies.csv"), ("sites", "sites.csv"),
            ("subjects", "subjects.csv"), ("visits", "visits.csv")]

def _upsert(df, schema, table, key, delete_not_matched_by_source=False):
    df.createOrReplaceTempView("v_src")
    spark.sql(f"CREATE TABLE IF NOT EXISTS {schema}.{table} USING DELTA "
             f"AS SELECT * FROM v_src WHERE 1=0")
    extra = " WHEN NOT MATCHED BY SOURCE THEN DELETE" if delete_not_matched_by_source else ""
    spark.sql(f"""MERGE INTO {schema}.{table} t USING v_src s ON t.{key} = s.{key}
                  WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *{extra}""")
    return spark.table(f"{schema}.{table}").count()

def stage_bronze():
    """Incremental append-only ingest; returns total new rows appended."""
    wm = spark.sql(f"""SELECT MAX(end_time) AS wm FROM {RUN_LOG_TABLE}
                       WHERE stage_name = 'Bronze_Ingestion' AND status = 'Success'"""
                   ).collect()[0]["wm"]
    total = 0
    for table, fname in DATASETS:
        cols = BUSINESS_COLUMNS[table]
        spark.sql(f"CREATE TABLE IF NOT EXISTS {BRONZE}.{table} "
                  f"({', '.join(c + ' STRING' for c in cols)}) USING DELTA")
        path = f"{LANDING_PATH}/{fname}"
        mtime = datetime.datetime.fromtimestamp(
            dbutils.fs.ls(path)[0].modificationTime / 1000.0, tz=datetime.UTC)
        # Spark TIMESTAMP reads back offset-naive; make the watermark tz-aware
        wm_utc = (wm.replace(tzinfo=datetime.UTC)
                  if (wm is not None and wm.tzinfo is None) else wm)
        if wm_utc is not None and mtime <= wm_utc:
            continue  # unchanged file -> skip entirely
        src_df = read_csv_all_strings(spark, path, cols)
        new_df = new_rows_only(src_df, spark.table(f"{BRONZE}.{table}"), cols)
        if (n := new_df.count()) > 0:
            col_sql = ", ".join(cols)
            new_df.createOrReplaceTempView("v_bronze_new")
            spark.sql(f"INSERT INTO {BRONZE}.{table} ({col_sql}) "
                      f"SELECT {col_sql} FROM v_bronze_new")
            total += n
    return total

def stage_silver():
    """Transform + upsert Silver, then run the DQ gate; returns row count."""
    ts = datetime.datetime.now(datetime.UTC).isoformat()
    counts = 0
    counts += _upsert(transform_studies(spark.table(f"{BRONZE}.studies"), ts), SILVER, "studies", "study_id")
    counts += _upsert(transform_sites(spark.table(f"{BRONZE}.sites"), ts), SILVER, "sites", "site_id")
    valid, quar = transform_subjects(spark.table(f"{BRONZE}.subjects"), ts)
    counts += _upsert(valid, SILVER, "subjects", "subject_id")
    counts += _upsert(quar, SILVER, "subjects_quarantine", "subject_id")
    visits, visits_q = transform_visits(spark.table(f"{BRONZE}.visits"), spark.table(f"{SILVER}.subjects"), ts)
    counts += _upsert(visits, SILVER, "visits", "visit_id", delete_not_matched_by_source=True)
    counts += _upsert(visits_q, SILVER, "visits_quarantine", "visit_id")
    checks = run_dq_checks(
        subjects=spark.table(f"{SILVER}.subjects"), visits=spark.table(f"{SILVER}.visits"),
        studies=spark.table(f"{SILVER}.studies"), sites=spark.table(f"{SILVER}.sites"))
    print(summarize(checks))
    if not all_passed(checks):
        raise RuntimeError("Silver DQ gate failed — see DQ summary above")
    return counts

def stage_gold():
    """Build + upsert both Gold KPI summaries; returns row count."""
    ts = datetime.datetime.now(datetime.UTC).isoformat()
    n1 = _upsert(build_study_kpi_summary(
        spark.table(f"{SILVER}.studies"), spark.table(f"{SILVER}.subjects"),
        spark.table(f"{SILVER}.visits"), gold_ts=ts), GOLD, "study_kpi_summary", "study_id")
    n2 = _upsert(build_site_kpi_summary(
        spark.table(f"{SILVER}.sites"), spark.table(f"{SILVER}.subjects"),
        spark.table(f"{SILVER}.visits"), gold_ts=ts), GOLD, "site_kpi_summary", "site_id")
    return n1 + n2

In [0]:
# --- Run the single route Bronze -> Silver -> Gold with stage logging ---
for _schema in (BRONZE, SILVER, GOLD):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {_schema}")

STAGES = [("Bronze_Ingestion", stage_bronze),
          ("Silver_Transformation", stage_silver),
          ("Gold_KPI_Analytics", stage_gold)]
results = {}
for stage_name, stage_fn in STAGES:
    STAGE_NAME, STAGE_START_TS = stage_name, datetime.datetime.now(datetime.UTC)
    _log_stage("Running")
    try:
        records = stage_fn()
        _log_stage("Success", records=records)
        results[stage_name] = records
    except Exception as e:
        _log_stage("Failed", error=e)
        raise RuntimeError(f"Pipeline aborted at {stage_name}") from e

print("=" * 60)
print(f"Pipeline run {RUN_ID} complete: {results}")